⚠️ **Privacidad:** las salidas se eliminaron porque los datos contienen información de estudiantes menores. Ejecute este cuaderno solo en un entorno institucional autorizado y no publique tablas con nombres, RUDE o fechas de nacimiento.


# Objetivo 1: Recoleccionar y depurar datos académicos históricos

En esta libreta se extrae la información de los **Excel (generados desde PDFs originales)**, se consolidan y se limpia la data creando la variable `rezago`.

In [ ]:
!pip install openpyxl pandas

In [ ]:
def limpiar_boletin(path):
    import pandas as pd

    raw = pd.read_excel(path, header=None, engine="openpyxl")

    start_row = None
    for i in range(len(raw)):
        row = raw.iloc[i].astype(str).str.lower()
        if row.str.contains("rude").any():
            start_row = i
            break

    if start_row is None:
        raise ValueError(f"No se detectó cabecera en {path}")

    df = pd.read_excel(path, skiprows=start_row, engine="openpyxl")

    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all")

    df.columns = df.columns.astype(str).str.strip().str.lower()

    rude_col = None
    for col in df.columns:
        if "rude" in col.replace(".", ""):
            rude_col = col
            break

    if not rude_col:
        raise ValueError(f"No se encontró RUDE en {path}")

    df = df.rename(columns={rude_col: "rude"})

    materias_map = {
        "pa": "com_lenguajes",
        "pa.1": "cs_sociales",
        "pa.2": "edu_fisica",
        "pa.3": "edu_musical",
        "pa.4": "art_plasticas",
        "pa.5": "matematica",
        "pa.6": "tec_tecnologica",
        "pa.7": "cs_naturales",
        "pa.8": "valores_religion"
    }

    df = df.rename(columns=materias_map)

    df.columns = (
        df.columns
        .str.replace(" ", "_")
        .str.replace("á","a").str.replace("é","e")
        .str.replace("í","i")
        .str.replace("ó","o")
        .str.replace("ú","u")
    )

    for col in materias_map.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[df["rude"].notna()]

    return df


In [ ]:
def limpiar_boletin_v2(path):
    import pandas as pd
    import os

    # 1. Leemos el archivo "crudo" para buscar cabeceras y metadatos
    raw = pd.read_excel(path, header=None, engine="openpyxl")

    # --- NUEVO: EXTRACCIÓN DE METADATOS ---
    # Basado en la estructura estandar del Ministerio que vimos en tu archivo:
    # Fila 0, Col 1 -> Gestión (Ej: 2022)
    # Fila 1, Col 3 -> Año Escolaridad (Ej: CUARTO)
    # Fila 1, Col 5 -> Paralelo (Ej: A)
    try:
        gestion_val = str(raw.iloc[0, 1]).strip()
        curso_val = str(raw.iloc[1, 3]).strip().upper()
        paralelo_val = str(raw.iloc[1, 5]).strip().upper()
    except Exception as e:
        print(f"Advertencia: No se pudieron extraer metadatos exactos de {path}. Se dejarán vacíos.")
        gestion_val, curso_val, paralelo_val = None, None, None
    # --------------------------------------

    # 2. Lógica original para encontrar dónde empieza la tabla
    start_row = None
    for i in range(len(raw)):
        # Convertimos a string para evitar error si hay nulos
        row = raw.iloc[i].astype(str).str.lower()
        if row.str.contains("rude").any():
            start_row = i
            break

    if start_row is None:
        raise ValueError(f"No se detectó cabecera en {path}")

    # 3. Leemos la tabla limpia desde la fila encontrada
    df = pd.read_excel(path, skiprows=start_row, engine="openpyxl")

    # Limpieza básica de filas/columnas vacías
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all")

    # Normalización de nombres de columnas
    df.columns = df.columns.astype(str).str.strip().str.lower()

    # Búsqueda de la columna RUDE
    rude_col = None
    for col in df.columns:
        if "rude" in col.replace(".", ""):
            rude_col = col
            break

    if not rude_col:
        raise ValueError(f"No se encontró RUDE en {path}")

    df = df.rename(columns={rude_col: "rude"})

    # Mapeo de materias
    materias_map = {
        "pa": "com_lenguajes",
        "pa.1": "cs_sociales",
        "pa.2": "edu_fisica",
        "pa.3": "edu_musical",
        "pa.4": "art_plasticas",
        "pa.5": "matematica",
        "pa.6": "tec_tecnologica",
        "pa.7": "cs_naturales",
        "pa.8": "valores_religion"
    }

    df = df.rename(columns=materias_map)

    # Limpieza de caracteres raros en los nombres de columna
    df.columns = (
        df.columns
        .str.replace(" ", "_")
        .str.replace("á","a").str.replace("é","e")
        .str.replace("í","i")
        .str.replace("ó","o")
        .str.replace("ú","u")
        .str.replace("\n", "") # Agregué esto por si hay saltos de linea
    )

    # Conversión a numérico de las notas
    for col in materias_map.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Filtrar filas sin RUDE
    df = df[df["rude"].notna()]

    # --- NUEVO: INYECCIÓN DE METADATOS AL DATAFRAME ---
    # Asignamos los valores extraídos al principio a nuevas columnas
    df.insert(0, 'gestion', gestion_val)
    df.insert(1, 'anio_escolaridad', curso_val)
    df.insert(2, 'paralelo', paralelo_val)
    
    # Opcional: Agregar el nombre del archivo origen para rastreo
    df['archivo_origen'] = os.path.basename(path)
    # --------------------------------------------------

    return df

In [ ]:
import glob, os, pandas as pd, re

BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "data/02_Datos_Extraidos_Excel")

files = glob.glob(os.path.join(DATA_DIR, "**", "*.xlsx"), recursive=True)
files = [f for f in files if not os.path.basename(f).startswith("~$")]

print("Archivos encontrados:", len(files))

dfs = []

for f in files:
    try:
        df = limpiar_boletin_v2(f)
        dfs.append(df)
    except Exception as e:
        print("⚠️ Error en:", f)
        print("   ", e)

if len(dfs) == 0:
    raise ValueError("No se pudo procesar ningún archivo")

dataset = pd.concat(dfs, ignore_index=True)
print("Total registros:", len(dataset))


In [ ]:
# // Alumnos rezagados
materias = [
    "com_lenguajes",
    "cs_sociales",
    "edu_fisica",
    "edu_musical",
    "art_plasticas",
    "matematica",
    "tec_tecnologica",
    "cs_naturales",
    "valores_religion"
]

dataset["rezago"] = (dataset[materias] < 51).any(axis=1).astype(int)

dataset["rezago"].value_counts()



In [ ]:
dataset.groupby("rezago")[materias].mean()


In [ ]:
# aniadir nueva columna

dataset["num_materias_reprobadas"] = (dataset[materias] < 51).sum(axis=1)
dataset["promedio_general"] = dataset[materias].mean(axis=1)

In [ ]:
dataset

In [ ]:
# Descargar dataset final en csv y excel
dataset.to_csv(
    "../data/03_Datasets_Procesados/primaria_dataset.csv",
    index=False,
    encoding="utf-8"
)
dataset.to_excel(
    "../data/03_Datasets_Procesados/primaria_dataset.xlsx",
    index=False
)

In [ ]:
dataset.shape
dataset.columns
dataset["rezago"].value_counts()


In [ ]:
print(dataset.columns.tolist())
